# H13 promoter/intergenic matrix inspection

Goal: inspect the two new H13 motif matrices before building the Elastic Net / enrichment pipeline.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROMOTER_FILE = Path("../promoter_TF_distance_weighted_H13_human_promoters_power_x1_d500.csv")
INTERGENIC_FILE = Path("../promoter_TF_distance_weighted_H13_human_intergenic_power_x1_d500.csv")

PROMOTER_FILE.exists(), INTERGENIC_FILE.exists()


In [ ]:
prom_head = pd.read_csv(PROMOTER_FILE, nrows=5)
inter_head = pd.read_csv(INTERGENIC_FILE, nrows=5)

print("Promoter preview shape:", prom_head.shape)
display(prom_head.head())

print("Intergenic preview shape:", inter_head.shape)
display(inter_head.head())


In [ ]:
prom_cols = pd.read_csv(PROMOTER_FILE, nrows=0).columns
inter_cols = pd.read_csv(INTERGENIC_FILE, nrows=0).columns

print("Promoter columns:", len(prom_cols))
print("Intergenic columns:", len(inter_cols))
print("Columns identical:", list(prom_cols) == list(inter_cols))

print("\nFirst 10 columns:")
print(prom_cols[:10].tolist())

print("\nLast 10 columns:")
print(prom_cols[-10:].tolist())


In [ ]:
keywords = ["tpm", "expr", "expression", "log", "rna", "count", "target"]

for name, cols in [("promoter", prom_cols), ("intergenic", inter_cols)]:
    hits = [c for c in cols if any(k in c.lower() for k in keywords)]
    print(name, "possible expression columns:", hits if hits else "NONE")


In [ ]:
for f in [PROMOTER_FILE, INTERGENIC_FILE]:
    n_rows = sum(1 for _ in open(f)) - 1
    print(f.name, "rows:", n_rows)


In [ ]:
prom = pd.read_csv(PROMOTER_FILE)
inter = pd.read_csv(INTERGENIC_FILE)

motif_cols = [c for c in prom.columns if c != "promoter_id"]

summary = pd.DataFrame({
    "dataset": ["promoter", "intergenic"],
    "rows": [len(prom), len(inter)],
    "columns": [prom.shape[1], inter.shape[1]],
    "motif_columns": [len(motif_cols), len(motif_cols)],
    "missing_values": [prom[motif_cols].isna().sum().sum(), inter[motif_cols].isna().sum().sum()],
    "negative_values": [(prom[motif_cols] < 0).sum().sum(), (inter[motif_cols] < 0).sum().sum()],
    "all_zero_rows": [
        (prom[motif_cols].sum(axis=1) == 0).sum(),
        (inter[motif_cols].sum(axis=1) == 0).sum()
    ],
    "duplicate_ids": [
        prom["promoter_id"].duplicated().sum(),
        inter["promoter_id"].duplicated().sum()
    ]
})

display(summary)


In [ ]:
prom["motif_sum"] = prom[motif_cols].sum(axis=1)
inter["motif_sum"] = inter[motif_cols].sum(axis=1)

display(pd.DataFrame({
    "promoter": prom["motif_sum"].describe(),
    "intergenic": inter["motif_sum"].describe()
}))


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(prom["motif_sum"], bins=80, alpha=0.6, label="promoter")
plt.hist(inter["motif_sum"], bins=80, alpha=0.6, label="intergenic")
plt.xlabel("Total motif signal per region")
plt.ylabel("Number of regions")
plt.legend()
plt.title("Distribution of total motif signal")
plt.show()


In [ ]:
top_prom = prom[["promoter_id", "motif_sum"]].sort_values("motif_sum", ascending=False).head(10)
top_inter = inter[["promoter_id", "motif_sum"]].sort_values("motif_sum", ascending=False).head(10)

print("Top promoter regions by motif sum")
display(top_prom)

print("Top intergenic regions by motif sum")
display(top_inter)


In [ ]:
motif_means = pd.DataFrame({
    "motif": motif_cols,
    "promoter_mean": prom[motif_cols].mean().values,
    "intergenic_mean": inter[motif_cols].mean().values
})

motif_means["intergenic_minus_promoter"] = motif_means["intergenic_mean"] - motif_means["promoter_mean"]
motif_means["intergenic_over_promoter"] = motif_means["intergenic_mean"] / motif_means["promoter_mean"].replace(0, np.nan)

display(motif_means.sort_values("intergenic_minus_promoter", ascending=False).head(20))
display(motif_means.sort_values("intergenic_minus_promoter", ascending=True).head(20))


## Conclusion notes

If no TPM/expression column appears above, these two files are motif-only matrices.

Next step: locate the H13 TPM/expression table or inspect the old notebook to see how TPM was merged.
